# Lab | Agent & Vector store


<br>

## Intro

In this lab you'll build an AI agent that knows when to consult *different* knowledge bases to answer a question — instead of relying on a single source of truth.

Here's what to expect:

1. **Follow a full worked demo** — We'll walk through every step together: ingesting the *state of the union* speech and the *Ruff* docs into two vector stores, wrapping each in a `RetrievalQA` tool, and building an agent that picks the right tool (or both!) depending on the question.

2. **Replicate it yourself with a new dataset** — Then, you'll swap in a dataset of your choice and rebuild the same pipeline, adapting the prompts and tools along the way.

By the end of this lab, you'll understand how to build multi-source AI agents and be able to apply the pattern to your own datasets.

<br>

## Combine agents and vector stores

Let's get into the demo. We'll wrap each vector store in a `RetrievalQA` chain and hand it to an agent as a `Tool`. The agent then decides, at each step, which tool to call based purely on its description — this is what lets it route between multiple knowledge sources.

There are two flavors of this pattern, both of which we'll try below:

- **Agent as reasoner** — the agent calls a tool and can keep reasoning afterward (e.g. to combine results from multiple sources).
- **Agent as router** (`return_direct=True`) — the agent just picks the right tool and returns its answer immediately, no extra reasoning.

<br>

## Install dependencies

Uncomment and run the cells below to install the dependencies required for this notebook.

Tip: Use a virtual environment to keep this project's dependencies isolated from your system Python and other projects.

In [2]:
!pip install "langchain<0.3" "langchain-core<0.3" "langchain-community<0.3" "langchain-openai<0.2"

  Using cached langchain-0.2.17-py3-none-any.whl.metadata (7.1 kB)
  Using cached langchain_core-0.2.43-py3-none-any.whl.metadata (6.2 kB)
  Using cached langchain_openai-0.1.25-py3-none-any.whl.metadata (2.6 kB)
  Using cached langchain_text_splitters-0.2.4-py3-none-any.whl.metadata (2.3 kB)
  Using cached langsmith-0.1.147-py3-none-any.whl.metadata (14 kB)
  Using cached openai-1.109.1-py3-none-any.whl.metadata (29 kB)
Using cached langchain-0.2.17-py3-none-any.whl (1.0 MB)
Using cached langchain_core-0.2.43-py3-none-any.whl (397 kB)
Using cached langchain_openai-0.1.25-py3-none-any.whl (51 kB)
Using cached langchain_text_splitters-0.2.4-py3-none-any.whl (25 kB)
Using cached langsmith-0.1.147-py3-none-any.whl (311 kB)
Using cached openai-1.109.1-py3-none-any.whl (948 kB)
  Attempting uninstall: openai
    Found existing installation: openai 3.42.0
    Uninstalling openai-3.42.0:
      Successfully uninstalled openai-3.42.0
  Attempting uninstall: langsmith
    Found existing installa

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-classic 1.0.8 requires langchain-core<2.0.0,>=1.4.4, but you have langchain-core 0.2.43 which is incompatible.
langchain-classic 1.0.8 requires langchain-text-splitters<2.0.0,>=1.1.2, but you have langchain-text-splitters 0.2.4 which is incompatible.
langgraph 1.2.19 requires langchain-core<2,>=1.4.7, but you have langchain-core 0.2.43 which is incompatible.
langgraph-prebuilt 1.1.1 requires langchain-core>=1.3.1, but you have langchain-core 0.2.43 which is incompatible.
langgraph-sdk 0.4.11 requires langchain-core<2,>=1.4.0, but you have langchain-core 0.2.43 which is incompatible.

[notice] A new release of pip is available: 24.0 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
!pip install python-dotenv==1.2.2 chromadb==1.5.9 beautifulsoup4==4.15.0


[notice] A new release of pip is available: 24.0 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


<br>

## Initial Setup

Before building anything, we need to load our API credentials, instantiate the LLM we'll use throughout the notebook, and locate the sample document we'll be querying.

In [2]:
from langchain.chains import RetrievalQA
from langchain_community.vectorstores import Chroma
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_text_splitters import CharacterTextSplitter
from langchain_community.document_loaders import TextLoader

In [3]:
import os
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())

OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY')


In [ ]:
llm = OpenAI( model="gpt-4o-mini", temperature=0, api_key=OPENAI_API_KEY)

<br>

Now let's ingest the state of the union speech: load the raw text, split it into manageable chunks, embed those chunks, and store them in a Chroma vector store.

In [5]:
doc_path =  "./datasets/state_of_the_union.txt"

In [ ]:
loader = TextLoader(doc_path)
documents = loader.load()
text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=0)
texts = text_splitter.split_documents(documents)

embeddings = OpenAIEmbeddings(api_key=OPENAI_API_KEY)

docsearch = Chroma.from_documents(texts, embeddings, collection_name="state-of-union")

<br>

## Adding a second knowledge source

To show how an agent can route between multiple tools, let's add a second vector store — this time built from the Ruff FAQ web page instead of a local file.

In [9]:
state_of_union = RetrievalQA.from_chain_type(
    llm=llm, chain_type="stuff", retriever=docsearch.as_retriever()
)

In [10]:
from langchain_community.document_loaders import WebBaseLoader

USER_AGENT environment variable not set, consider setting it to identify your requests.


In [11]:
loader = WebBaseLoader("https://beta.ruff.rs/docs/faq/")

In [12]:
docs = loader.load()
ruff_texts = text_splitter.split_documents(docs)
ruff_db = Chroma.from_documents(ruff_texts, embeddings, collection_name="ruff")
ruff = RetrievalQA.from_chain_type(
    llm=llm, chain_type="stuff", retriever=ruff_db.as_retriever()
)

Created a chunk of size 2209, which is longer than the specified 1000
Created a chunk of size 3190, which is longer than the specified 1000
Created a chunk of size 1017, which is longer than the specified 1000
Created a chunk of size 2313, which is longer than the specified 1000


<br>

## Create the Agent

With both `RetrievalQA` chains ready, we wrap each one in a `Tool` (giving it a name and a description the agent will use to decide when to call it), then hand both tools to an agent.

In [13]:
# Import things that are needed generically
from langchain.agents import AgentType, Tool, initialize_agent
from langchain_openai import OpenAI

<br>

Let's try it out — first with a question only the state of the union tool can answer, then one only Ruff can answer. Watch the verbose output to see which tool the agent picks each time.

In [14]:
tools = [
    Tool(
        name="State of Union QA System",
        func=state_of_union.run,
        description="useful for when you need to answer questions about the most recent state of the union address. Input should be a fully formed question.",
    ),
    Tool(
        name="Ruff QA System",
        func=ruff.run,
        description="useful for when you need to answer questions about ruff (a python linter). Input should be a fully formed question.",
    ),
]

In [15]:
# Construct the agent. We will use the default agent type here.
# See documentation for a full list of options.
agent = initialize_agent(
    tools, llm, agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION, verbose=True
)

C:\Users\rayha\AppData\Local\Temp\ipykernel_28004\1834837320.py:3: LangChainDeprecationWarning: The function `initialize_agent` was deprecated in LangChain 0.1.0 and will be removed in 1.0. Use Use new agent constructor methods like create_react_agent, create_json_agent, create_structured_chat_agent, etc. instead.
  agent = initialize_agent(


In [16]:
agent.invoke(
    "What did biden say about ketanji brown jackson in the state of the union address?"
)

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


I need to find out what President Biden said about Ketanji Brown Jackson in the most recent State of the Union address. 
Action: State of Union QA System
Action Input: What did Biden say about Ketanji Brown Jackson in the State of the Union address?
Observation: In the State of the Union address, Biden mentioned that he nominated Circuit Court of Appeals Judge Ketanji Brown Jackson four days prior to the speech. He described her as one of the nation’s top legal minds who will continue Justice Stephen Breyer’s legacy of excellence.
Thought:I now know the final answer.  
Final Answer: In the State of the Union address, Biden mentioned that he nominated Circuit Court of Appeals Judge Ketanji Brown Jackson four days prior to the speech, describing her as one of the nation’s top legal minds who will continue Justice Stephen Breyer’s legacy of excellence.

> Finished chain.


{'input': 'What did biden say about ketanji brown jackson in the state of the union address?',
 'output': 'In the State of the Union address, Biden mentioned that he nominated Circuit Court of Appeals Judge Ketanji Brown Jackson four days prior to the speech, describing her as one of the nation’s top legal minds who will continue Justice Stephen Breyer’s legacy of excellence.'}

In [17]:
agent.invoke("Why use ruff over flake8?")

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


I need to gather information about the advantages of using ruff compared to flake8, which is a popular Python linter. This will help me provide a comprehensive answer to the question. 
Action: Ruff QA System
Action Input: "What are the advantages of using ruff over flake8?"
Observation: The advantages of using Ruff over Flake8 include:

1. **Performance**: Ruff is designed to be faster than Flake8, providing quicker feedback on lint violations.

2. **Comprehensive Rule Set**: Ruff implements over 900 rules, including many popular Flake8 plugins re-implemented natively, which allows for a broader range of linting capabilities.

3. **Automatic Fixing**: Ruff is capable of automatically fixing its own lint violations, which Flake8 does not support.

4. **No Custom Rules**: Ruff does not support custom lint rules, which simplifies the setup and usage compared to Flake8, where users may need to manage multiple plugins.

5. **Compatibility with Formatters**: Ruff's linter is compatible with 

{'input': 'Why use ruff over flake8?',
 'output': 'The advantages of using Ruff over Flake8 include better performance, a comprehensive rule set with over 900 rules, automatic fixing of lint violations, simplified configuration without custom rules, compatibility with formatters like Black, and the ability to serve as a drop-in replacement for Flake8. Overall, Ruff provides a more modern and efficient approach to linting Python code.'}

## Use the Agent solely as a router

<br>

You can also set `return_direct=True` if you intend to use the agent as a router and just want to directly return the result of the RetrievalQAChain.

Notice that in the above examples the agent did some extra work after querying the RetrievalQAChain. You can avoid that and just return the result directly.

In [18]:
tools = [
    Tool(
        name="State of Union QA System",
        func=state_of_union.run,
        description="useful for when you need to answer questions about the most recent state of the union address. Input should be a fully formed question.",
        return_direct=True,
    ),
    Tool(
        name="Ruff QA System",
        func=ruff.run,
        description="useful for when you need to answer questions about ruff (a python linter). Input should be a fully formed question.",
        return_direct=True,
    ),
]

In [19]:
agent = initialize_agent(
    tools, llm, agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION, verbose=True
)

In [20]:
agent.invoke(
    "What did biden say about ketanji brown jackson in the state of the union address?"
)

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


I need to find out what President Biden said about Ketanji Brown Jackson in the most recent State of the Union address. 
Action: State of Union QA System
Action Input: What did Biden say about Ketanji Brown Jackson in the State of the Union address?
Observation: In the State of the Union address, Biden mentioned that he nominated Circuit Court of Appeals Judge Ketanji Brown Jackson four days prior to the speech. He described her as one of the nation’s top legal minds who will continue Justice Stephen Breyer’s legacy of excellence.


> Finished chain.


{'input': 'What did biden say about ketanji brown jackson in the state of the union address?',
 'output': 'In the State of the Union address, Biden mentioned that he nominated Circuit Court of Appeals Judge Ketanji Brown Jackson four days prior to the speech. He described her as one of the nation’s top legal minds who will continue Justice Stephen Breyer’s legacy of excellence.'}

In [21]:
agent.invoke("Why use ruff over flake8?")

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


I need to gather information about the advantages of using ruff compared to flake8, which is a popular Python linter. This will help me provide a comprehensive answer to the question. 
Action: Ruff QA System
Action Input: "What are the advantages of using ruff over flake8?"
Observation: The advantages of using Ruff over Flake8 include:

1. **Performance**: Ruff is designed to be faster than Flake8, providing quicker feedback on lint violations.

2. **Comprehensive Rule Set**: Ruff implements over 900 rules, including many popular Flake8 plugins re-implemented natively, which allows for a broader range of linting checks.

3. **Automatic Fixing**: Ruff is capable of automatically fixing its own lint violations, which Flake8 does not support.

4. **No Custom Rules**: Ruff does not support custom lint rules, but this means that it has a consistent rule set without the potential conflicts that can arise from various plugins in Flake8.

5. **Compatibility with Formatters**: Ruff is designed 

{'input': 'Why use ruff over flake8?',
 'output': 'The advantages of using Ruff over Flake8 include:\n\n1. **Performance**: Ruff is designed to be faster than Flake8, providing quicker feedback on lint violations.\n\n2. **Comprehensive Rule Set**: Ruff implements over 900 rules, including many popular Flake8 plugins re-implemented natively, which allows for a broader range of linting checks.\n\n3. **Automatic Fixing**: Ruff is capable of automatically fixing its own lint violations, which Flake8 does not support.\n\n4. **No Custom Rules**: Ruff does not support custom lint rules, but this means that it has a consistent rule set without the potential conflicts that can arise from various plugins in Flake8.\n\n5. **Compatibility with Formatters**: Ruff is designed to work well alongside formatters like Black, and it can defer stylistic rules that are handled by automated formatting.\n\n6. **Drop-in Replacement**: Under certain conditions (without or with a small number of plugins, alongs

<br>

## Multi-Hop vector store reasoning

Because vector stores are easily usable as tools in agents, it is easy to use answer multi-hop questions that depend on vector stores using the existing agent framework.

In [22]:
tools = [
    Tool(
        name="State of Union QA System",
        func=state_of_union.run,
        description="useful for when you need to answer questions about the most recent state of the union address. Input should be a fully formed question, not referencing any obscure pronouns from the conversation before.",
    ),
    Tool(
        name="Ruff QA System",
        func=ruff.run,
        description="useful for when you need to answer questions about ruff (a python linter). Input should be a fully formed question, not referencing any obscure pronouns from the conversation before.",
    ),
]

In [23]:
# Construct the agent. We will use the default agent type here.
# See documentation for a full list of options.
agent = initialize_agent(
    tools, llm, agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION, verbose=True
)

In [24]:
agent.invoke(
    "What tool does ruff use to run over Jupyter Notebooks? Did the president mention that tool in the state of the union?"
)

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


I need to find out what tool Ruff uses to run over Jupyter Notebooks and whether the president mentioned that tool in the State of the Union address. I will first check the Ruff QA System for information about Ruff's functionality and then check the State of Union QA System for any mention of that tool in the State of the Union address.

Action: Ruff QA System  
Action Input: What tool does Ruff use to run over Jupyter Notebooks?  
Observation: Ruff integrates with nbQA, a tool for running linters and code formatters over Jupyter Notebooks.
Thought:I have found that Ruff integrates with nbQA to run over Jupyter Notebooks. Now, I need to check if the president mentioned nbQA in the State of the Union address.

Action: State of Union QA System  
Action Input: Did the president mention nbQA in the State of the Union address?  
Observation: I don't know.
Thought:I need to find out if the president mentioned nbQA in the State of the Union address. Since I don't have the information yet, I w

{'input': 'What tool does ruff use to run over Jupyter Notebooks? Did the president mention that tool in the state of the union?',
 'output': 'Ruff uses nbQA to run over Jupyter Notebooks, but the president did not mention that tool in the State of the Union address.'}

<br><br>

---

<br>

## 🚀 Your turn

Time to make this lab your own! 

Replace the `state_of_the_union.txt` dataset with something you'd actually enjoy chatting with. A great place to start is the [sonnets.txt dataset](https://github.com/martin-gorner/tensorflow-rnn-shakespeare/blob/master/shakespeare/sonnets.txt) —or any other .txt file from that repository. Of course, you're not limited to those options. Pick any text that interests you and see how your chatbot responds.

Here's what to do:

1. **Get your data** — Download your chosen `.txt` file into this project folder (or point `TextLoader` at it directly).
2. **Rebuild the vector store** — Load, split, and embed your new document, then create a fresh `RetrievalQA` chain for it (give it a descriptive `collection_name`!).
3. **Rewrite the tool description** — Update the `Tool`'s `name` and `description` so the agent knows *when* it should reach for this new tool instead of the Ruff or state-of-the-union ones.
4. **Rebuild the agent** — Combine your new tool with the existing Ruff tool (or drop it if you'd rather keep just your new dataset + one other source).
5. **Put it to the test** — Ask your agent:
   - A direct question that only your new dataset can answer.
   - A question that only the Ruff tool can answer.
   - A multi-hop question that requires combining *both* tools' knowledge, like the Jupyter/Ruff example above.
6. **Reflect** — In a markdown cell, briefly note whether the agent picked the right tool(s) each time, and what happened when you set `return_direct=True` vs. not.

⭐️ **Bonus points:**
- Instead of modifying this same file, create a new file `solution.ipynb` and replicate the process from scratch.

💡 **Tip:** 
- Watch the `verbose=True` agent logs closely — they show you the agent's reasoning step by step, which is the best way to understand *why* it picked a particular tool.



In [25]:
# Your turn: load Shakespeare's sonnets from datasets/sonnets.txt
sonnets_path = "./datasets/sonnets.txt"
sonnets_loader = TextLoader(sonnets_path, encoding="utf-8")
sonnets_documents = sonnets_loader.load()
sonnets_texts = text_splitter.split_documents(sonnets_documents)
print("Sonnets chunks:", len(sonnets_texts))

Sonnets chunks: 154


In [26]:
# Rebuild vector store and RetrievalQA chain using the ORIGINAL LangChain pattern
sonnets_db = Chroma.from_documents(
    sonnets_texts, embeddings, collection_name="shakespeare-sonnets"
)
sonnets = RetrievalQA.from_chain_type(
    llm=llm, chain_type="stuff", retriever=sonnets_db.as_retriever()
)

In [27]:
# Rewrite tool names/descriptions and rebuild the original initialize_agent
# Keep Ruff as the second knowledge source.
tools = [
    Tool(
        name="Shakespeare Sonnets QA System",
        func=sonnets.run,
        description="useful for questions about Shakespeare's sonnets, their themes, and their language. Input should be a fully formed question.",
    ),
    Tool(
        name="Ruff QA System",
        func=ruff.run,
        description="useful for questions about Ruff, the Python linter and formatter. Input should be a fully formed question.",
    ),
]
agent = initialize_agent(
    tools, llm, agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION, verbose=True,
    handle_parsing_errors=True,
)

In [28]:
# Test 1: Question only the Shakespeare sonnets dataset should answer
result_sonnets = agent.invoke({"input": "What comparison does Shakespeare make in Sonnet 18 about a summer's day?"})
print(result_sonnets["output"])

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


I need to find out what comparison Shakespeare makes in Sonnet 18 regarding a summer's day. This sonnet is well-known for its exploration of beauty and nature, so I will refer to the Shakespeare Sonnets QA System for detailed insights.  
Action: Shakespeare Sonnets QA System  
Action Input: What comparison does Shakespeare make in Sonnet 18 about a summer's day?  
Observation: In Sonnet 18, Shakespeare compares the beloved to a summer's day, stating that the beloved is "more lovely and more temperate" than summer. He notes that summer can be too hot and is often short-lived, while the beauty of the beloved is eternal and will not fade. The sonnet emphasizes that the beloved's beauty will live on through the poem itself, contrasting the fleeting nature of summer with the lasting nature of the beloved's beauty.
Thought:I now know the final answer.  
Final Answer: In Sonnet 18, Shakespeare compares the beloved to a summer's day, stating that the beloved is "more lovely and more temperate"

In [29]:
# Test 2: Question only the Ruff FAQ should answer
result_ruff = agent.invoke({"input": "What is Ruff, and how does it relate to Flake8?"})
print(result_ruff["output"])

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


I need to gather information about Ruff and its relationship with Flake8. This will involve understanding what Ruff is and how it compares or connects to Flake8, which is a well-known Python linter. 
Action: Ruff QA System
Action Input: What is Ruff, and how does it relate to Flake8?
Observation: Ruff is a linter for Python that can be used as a drop-in replacement for Flake8 under certain conditions. It implements over 900 rules, including all of the F rules from Pyflakes and a subset of the E and W rules from pycodestyle. Ruff also re-implements some popular Flake8 plugins natively, which helps minimize conflicts across plugins.

One key difference between Ruff and Flake8 is that Ruff does not support custom lint rules, while Flake8 allows for plugins (called "checkers"). Ruff is designed to provide faster feedback on lint violations and can automatically fix its own lint violations, which Flake8 does not do. Additionally, Ruff's primary limitation compared to Flake8 is that it does 

In [30]:
# Test 3: Multi-hop question requiring BOTH knowledge sources
result_both = agent.invoke({
    "input": "What does Sonnet 18 compare someone to, and can Ruff lint Python files that analyze such poems? Answer both parts using the relevant tools."
})
print(result_both["output"])

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


I need to answer two parts of the question: the first part regarding Sonnet 18 and the second part about Ruff's capabilities. I will first use the Shakespeare Sonnets QA System to find out what Sonnet 18 compares someone to. Then, I will use the Ruff QA System to determine if it can lint Python files that analyze poems.

Action: Shakespeare Sonnets QA System  
Action Input: What does Sonnet 18 compare someone to?  
Observation: Sonnet 18 compares someone to a summer's day.
Thought:I have the answer to the first part of the question. Now, I will proceed to the second part regarding Ruff's capabilities.

Action: Ruff QA System  
Action Input: Can Ruff lint Python files that analyze poems?  
Observation: I don't know.
Thought:I need to clarify whether Ruff can lint Python files that analyze poems. Since I don't have a definitive answer from the previous action, I will look for more information about Ruff's capabilities.

Action: Ruff QA System  
Action Input: What are the capabilities of 

In [31]:
# Compare the original agent behavior with return_direct=True
# With return_direct=True, the agent stops and returns the chosen tool's output.
router_tools = [
    Tool(
        name="Shakespeare Sonnets QA System",
        func=sonnets.run,
        description="useful for questions about Shakespeare's sonnets and poetry. Input should be a fully formed question.",
        return_direct=True,
    ),
    Tool(
        name="Ruff QA System",
        func=ruff.run,
        description="useful for questions about Ruff, a Python linter. Input should be a fully formed question.",
        return_direct=True,
    ),
]
router_agent = initialize_agent(
    router_tools, llm, agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True, handle_parsing_errors=True,
)
router_result = router_agent.invoke({"input": "What does Shakespeare compare the subject to in Sonnet 18?"})
print(router_result["output"])

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


In Sonnet 18, Shakespeare famously compares the subject to a summer's day. I will use the Shakespeare Sonnets QA System to confirm this and gather more details about the comparison.  
Action: Shakespeare Sonnets QA System  
Action Input: "What does Shakespeare compare the subject to in Sonnet 18?"  
Observation: In Sonnet 18, Shakespeare compares the subject to a summer's day.


> Finished chain.
In Sonnet 18, Shakespeare compares the subject to a summer's day.
